# PortfolioPilot - 01 Cleaning and splitting

**Run the cells in order, top to bottom.** Install the package first (`pip install -e .` from the project root).

Rule for M3: the test sets are created in notebook 01 and then never explored. Every statistic, cap,
table and figure uses **training data only**.

Raw data in `data/raw/` is read-only. Everything written goes to `data/processed/`.

## Step 1 - Setting up

In [1]:
import numpy as np
import pandas as pd

from portfolio_pilot.config import Results, ensure_output_dirs, load_config

cfg = load_config()
ensure_output_dirs(cfg)
P, SEED = cfg["paths"], cfg["seed"]
results = Results(P["numbers_file"])    # every count quoted in the report; saved in the last cell
np.random.seed(SEED)

## Step 2 - Load the SCF and check its structure

In [2]:
from portfolio_pilot.clean import load_scf

full = load_scf(P["scf_file"], cfg["scf"]["columns"], results)
print(results["raw"])
full.head()

{'rows': 21800, 'columns': 362, 'households': 4360, 'implicates': [1, 2, 3, 4, 5]}


,household_id,record_id,weight,age,income,networth,liq,debt,equity,fin,yes_risk,no_risk,implicate
0,1,11,1241.701712,39,317351.35135,2434270.0,80300,250000.0,120150.0,215300,0,0,1
1,1,12,1247.038677,39,317351.35135,2208400.0,80300,250000.0,113800.0,229300,0,0,2
2,1,13,1234.428835,39,317351.35135,2128300.0,80300,250000.0,75300.0,209300,0,0,3
3,1,14,1267.530314,39,317351.35135,1742300.0,80300,250000.0,72800.0,212300,0,0,4
4,1,15,911.408367,39,317351.35135,1517300.0,80300,250000.0,73600.0,228300,0,0,5


## Step 3 - Cleaning: duplicates, missing values, impossible values

In [3]:
from portfolio_pilot.clean import clean_scf

full, hh = clean_scf(full, cfg["scf"]["implicate"], results)
for s in ["duplicates", "implicate_1", "missing", "errors", "genuine"]:
    print(s, results[s])
hh.to_csv(P["processed_dir"] / "scf_clean.csv", index=False)

duplicates {'exact_duplicate_rows': 0, 'duplicate_record_ids': 0}
implicate_1 {'rows': 4360, 'rows_after_cleaning': 4360}
missing {'by_column': {}, 'total': 0}
errors {'age_under_18_or_over_100': 0, 'weight_not_positive': 0, 'liq_negative': 0, 'debt_negative': 0, 'equity_negative': 0, 'fin_negative': 0, 'equity_more_than_fin': 0, 'liq_more_than_fin': 0, 'both_risk_flags_1': 0, 'rows_dropped': 0}
genuine {'income_zero': 23, 'income_negative': 0, 'networth_negative': 330, 'fin_zero': 53, 'debt_zero': 1115}


## Step 4 - Split train / validation / test (before any exploration)
Stratified on age band x risk willingness. Split by household, so implicates 2-5 can follow the same IDs later.

In [4]:
from portfolio_pilot.split import save_split_ids, split_households

parts = split_households(hh, cfg["scf"]["split"], SEED, cfg["scf"]["min_stratum_size"], results)
save_split_ids(parts, P["processed_dir"] / "scf_split_ids.csv")
results["split"]

{'n_strata': 18,
 'train_households': 3052,
 'train_weight_share': np.float64(0.7033403995202065),
 'val_households': 654,
 'val_weight_share': np.float64(0.1441687614754587),
 'test_households': 654,
 'test_weight_share': np.float64(0.15249083900433474)}

## Step 5 - Implicate structure check (training households only)

In [5]:
tr_full = full[full["household_id"].isin(parts["train"]["household_id"])]
g = tr_full.groupby("household_id")
results.note("implicates", "train_households_without_5_rows", int((g.size() != 5).sum()))
for c in ["income", "networth", "liq", "debt"]:
    results.note("implicates", f"share_where_{c}_varies", float((g[c].nunique() > 1).mean()))
results["implicates"]

{'train_households_without_5_rows': 0,
 'share_where_income_varies': 0.6055045871559633,
 'share_where_networth_varies': 0.9121887287024901,
 'share_where_liq_varies': 0.4384010484927916,
 'share_where_debt_varies': 0.42660550458715596}

## Step 6 - Load the ETF data and check its structure
If the file is missing, run `python -m portfolio_pilot.download_etf` from the project root.

In [6]:
from portfolio_pilot.clean import load_etf_prices

TICKERS = cfg["etf"]["tickers"]
wide = load_etf_prices(P["etf_prices_file"], TICKERS, results)
print(results["etf_raw"])
wide.head()

{'rows_trading_days': 2514, 'columns': 9, 'date_range': ['2016-01-04', '2025-12-31'], 'tickers_missing': [], 'unexpected_columns': [], 'missing_by_ticker': {'AGG': 0, 'DBC': 0, 'GLD': 0, 'SPY': 0, 'TIP': 0, 'TLT': 0, 'VEA': 0, 'VNQ': 0, 'VWO': 0}, 'first_valid_date': {'AGG': '2016-01-04', 'DBC': '2016-01-04', 'GLD': '2016-01-04', 'SPY': '2016-01-04', 'TIP': '2016-01-04', 'TLT': '2016-01-04', 'VEA': '2016-01-04', 'VNQ': '2016-01-04', 'VWO': '2016-01-04'}}


,AGG,DBC,GLD,SPY,TIP,TLT,VEA,VNQ,VWO
date,,,,,,,,,
2016-01-04,79.657730,11.210504,102.889999,169.051727,80.085335,89.580688,26.352303,51.486633,23.626696
2016-01-05,79.694603,11.100596,103.180000,169.337738,80.034424,89.219223,26.323164,52.480152,23.678732
2016-01-06,79.997070,10.923055,104.669998,167.201660,80.288979,90.421516,25.842310,52.336365,23.254976
2016-01-07,79.989708,10.872327,106.150002,163.190201,80.223534,90.583817,25.354166,51.316681,22.504095
2016-01-08,80.166794,10.821601,105.680000,161.398911,80.288979,90.989464,25.040882,50.630360,22.303364


## Step 7 - Clean ETF prices, compute returns, split by time

In [7]:
from portfolio_pilot.clean import clean_etf_prices, etf_wide_to_long
from portfolio_pilot.features import daily_log_returns
from portfolio_pilot.split import split_returns_by_time

prices = clean_etf_prices(etf_wide_to_long(wide), TICKERS, results)
ret = daily_log_returns(prices)

# train: 2016 to 2021 | validation: 2022 to 2023 | test: 2024 to 2025
etf_parts = split_returns_by_time(ret, cfg["etf"]["train_end"], cfg["etf"]["val_end"], results)
for name, d in etf_parts.items():
    d.to_csv(P["processed_dir"] / f"etf_returns_{name}.csv")

results["etf"], results["etf_split"]

({'rows': 22626,
  'tickers_missing': [],
  'duplicate_date_ticker': 0,
  'missing_prices': 0,
  'non_positive_prices': 0,
  'missing_days_by_ticker': {},
  'cells_forward_filled': 0,
  'trading_days': 2514,
  'date_range': ['2016-01-04', '2025-12-31']},
 {'train': {'days': 1510, 'from': '2016-01-05', 'to': '2021-12-31'},
  'val': {'days': 501, 'from': '2022-01-03', 'to': '2023-12-29'},
  'test': {'days': 502, 'from': '2024-01-02', 'to': '2025-12-31'}})

## Save the numbers quoted in the report

In [8]:
results.save()
print("Saved", P["numbers_file"])

Saved C:\Users\jp\Documents\DAE\7275\Pilot\Portfolio-Pilot\reports\numbers.json
